In [ ]:
import boto3
import awswrangler as wr

import io
import os
from pathlib import Path
import yaml

# read credentials stored in yaml file into data dictionary
searching_file = 'credentials.yml'
lookup_folder=r'C:\MySpace\Working\DE01\data_engineering_practices'


def lookup_file_path (searching_file, lookup_folder):
    for root, dirs, files in os.walk(lookup_folder):
        for name in files:
            if name == searching_file:  
                return os.path.abspath(os.path.join(root, name))
            

# read credentials stored in yaml file into data dictionary
config_file_path = lookup_file_path(searching_file,lookup_folder)
# print(config_file_path)
with open(config_file_path, 'r') as f:
    try:
        config_data = yaml.safe_load(f)
        # print(config_data)
    except yaml.YAMLError as exc:
        print(exc)

# get Credentials
aws_access_key_id = config_data['aws']['aws_access_key_id']
aws_secret_access_key = config_data['aws']['aws_secret_access_key']

# S3 data source Location
S3_SOURCE_BUCKET = 'cm-aws-s3-data-source'
S3_SOURCE_FOLDER_PATH = 'organization/sales'

# create a authorized session using boto3 + credentials
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)

# create s3 client to interact with AWS S3 (boto3 will automatically use configured credentials)
s3 = session.client(service_name='s3')

C:\MySpace\Working\DE01\data_engineering_practices\module_2_deep_dive_into_data_engineering\02_data_ingestion\credentials.yml


In [53]:
# get current date in timezone
import pytz
from datetime import datetime
today = datetime.now(tz=pytz.timezone('Australia/Sydney'))
year, month, day = today.strftime('%Y'), today.strftime('%m'), today.strftime('%d')

# Define destination bucket name and upload file folder
S3_DES_BUCKET_NAME = 'cm-aws-s3-destination'
S3_DES_FILE_PATH = f"k1/Angie/sales/ingestion_date={year}-{month}-{day}" # The destination path within S3 bucket

In [ ]:
# download single file from S3 to local machine
file_name='coffee_sales_202403.csv'
# define the file's S3 key,
OBJECT_KEY = f'{S3_SOURCE_FOLDER_PATH}/{file_name}' # The full path to the file within the S3 bucket

# define download destination
download_folder='./downloaded_data/'
os.makedirs(download_folder, exist_ok=True)
LOCAL_FILE_PATH = f'{download_folder}/{file_name}' # The destination path on local machine

# download file from S3
try:
    s3.download_file(S3_SOURCE_BUCKET, OBJECT_KEY, LOCAL_FILE_PATH)
    print(f"Successfully downloaded {OBJECT_KEY} to {LOCAL_FILE_PATH}")
except Exception as e:
    print(f"Error downloading file: {e}")

# upload file from local machine to S3 bucket
try:
    s3.upload_file(LOCAL_FILE_PATH, S3_DES_BUCKET_NAME, f'{S3_DES_FILE_PATH}/{file_name}')
    print(f"'{LOCAL_FILE_PATH}' successfully uploaded to '{S3_DES_BUCKET_NAME}/{S3_DES_FILE_PATH}/{file_name}'")
except Exception as e:
    print(f"Error uploading file: {e}")


In [ ]:
# download single file from S3 into memory
from io import BytesIO

# Use a BytesIO object or a local file opened in binary write mode
"""#Use a local file opened in binary mode:
with open('local_filename.txt', 'wb') as f:
    s3.download_fileobj(S3_SOURCE_BUCKET, OBJECT_KEY, f)
print(f"Successfully downloaded {OBJECT_KEY} to local memory/object")
"""
# download single file from S3 bucket into memory using download_fileobj
# Create a BytesIO object to act as an in-memory file
in_memory_file = BytesIO()

try:
    # Download the file to the in-memory object
    s3.download_fileobj(Bucket=S3_SOURCE_BUCKET, Key=OBJECT_KEY, Fileobj=in_memory_file)
    
    # After download, the file pointer is at the end of the data. 
    # Seek to the beginning to read it back
    in_memory_file.seek(0)
    
    # Read the content
    object_content = in_memory_file.read()

    # The content is in bytes. Decode if necessary.
    # text_content = object_content.decode('utf-8')
    
    print(f"Successfully downloaded {OBJECT_KEY} into memory")

       
except Exception as e:
    print(f"Error downloading file: {e}")

try:
    # Upload file to S3 using upload_fileobj 
    s3.upload_fileobj(in_memory_file, S3_DES_BUCKET_NAME, f'{S3_DES_FILE_PATH}/{file_name}')
    print(f"Data from memory successfully uploaded to '{S3_DES_BUCKET_NAME}/{S3_DES_FILE_PATH}/{file_name}'")
except Exception as e:
    print(f"Error uploading file: {e}")

Full Load - Snapshot

In [ ]:
# HINTS
# 1. list_objects_v2 >> List all objects
# 2. download_file >> download file
# 3. Download all files to "destination/sales/snapshot"

# download multiple files from S3 bucket

LOCAL_DESTINATION = './destination/sales/snapshot'

# AWS operations, like list_objects_v2 for S3, limit results to a certain number (e.g., 1000 objects) per request. 
# Paginators abstract the process of making these repeated calls to fetch all results.
paginator = s3.get_paginator('list_objects_v2')
# Use the paginator to iterate over pages of results for a specific bucket
pages = paginator.paginate(Bucket=S3_SOURCE_BUCKET, Prefix=S3_SOURCE_FOLDER_PATH) #prefix: Download only from a specific "folder"

for page in pages:
    if 'Contents' in page:
        for obj in page['Contents']:
            s3_key = obj['Key']
            # Create the local directory structure            
            local_file_path = Path(LOCAL_DESTINATION) / Path(s3_key)
            # Ensure parent directory exists
            local_file_path.parent.mkdir(parents=True, exist_ok=True)
            
            # Only download if it's a file (not an S3 folder marker ending in '/')
            if not s3_key.endswith('/'):
                print(f"Downloading: {s3_key} to {local_file_path}")
                s3.download_file(S3_SOURCE_BUCKET, s3_key, str(local_file_path))

# upload multiple files to S3 bucket
# S3 bucket details

import glob

# Find all files in the local directory
# Returns a list of paths matching the pathname pattern (filter specific file types, e.g., '*.csv' or all files '*')
files_to_upload = glob.glob(os.path.join(LOCAL_DESTINATION, '*'))

print(f"{len(files_to_upload)} files to upload.")

for local_file_path in files_to_upload:
    # Extract just the filename from the full path
    file_name = os.path.basename(local_file_path)

    # Construct the S3 object key (path in S3)
    s3_key = f"{S3_DES_FILE_PATH}/{file_name}" if S3_DES_FILE_PATH else file_name

    try:
        # Upload the file
        s3.upload_file(local_file_path, S3_DES_BUCKET_NAME, s3_key)
        print(f"Successfully uploaded {file_name} to s3://{S3_DES_BUCKET_NAME}/{s3_key}")
    except Exception as e:
        print(f"Error uploading {file_name}: {e}")

print("All uploads completed!")

"""
# uploading to destination S3 bucket
for local_file_path in files_to_upload:
    # Extract just the filename from the full path
    file_name = os.path.basename(local_file_path)
    # Create an in-memory file-like object (use BytesIO for binary, StringIO for text)
    # The 'rb' (read binary) mode is required for upload_fileobj
    file_object = io.BytesIO(file_name.encode('utf-8')) 

    try:
# Use upload_fileobj, which manages multipart uploads for large objects automatically
        s3.upload_fileobj(
            file_object,
            S3_DES_BUCKET_NAME,
                f'{S3_DES_FILE_PATH}/{file_name}' # This will be the key (path/filename) in S3
        )
        print(f"Successfully uploaded {file_name} to {S3_DES_BUCKET_NAME}")
    except Exception as e:
        print(f"Error uploading {file_name}: {e}")
"""

Full Load - Partition by Ingestion Date

In [ ]:
# HINTS
# 1. list_objects_v2 >> List all objects
# 2. download_file >> download file
# 3. Download all files to "destination/sales/<ingestion_date>"
# Paginators abstract the process of making these repeated calls to fetch all results.
paginator = s3.get_paginator('list_objects_v2')
# Use the paginator to iterate over pages of results for a specific bucket
pages = paginator.paginate(Bucket=S3_SOURCE_BUCKET, Prefix=S3_SOURCE_FOLDER_PATH) #prefix: Download only from a specific "folder"

for page in pages:
    if 'Contents' in page:
        for obj in page['Contents']:
            s3_key = obj['Key']
            # Create the local directory structure            
            local_file_path = Path(LOCAL_DESTINATION) / Path(s3_key)
            # Ensure parent directory exists
            local_file_path.parent.mkdir(parents=True, exist_ok=True)
            
            # Only download if it's a file (not an S3 folder marker ending in '/')
            if not s3_key.endswith('/'):
                print(f"Downloading: {s3_key} to {local_file_path}")
                print(str(obj['LastModified']))
                s3.download_file(S3_SOURCE_BUCKET, s3_key, str(local_file_path))

# upload multiple files to S3 bucket
# S3 bucket details

import glob

# Find all files in the local directory
# You can use glob patterns to filter specific file types, e.g., '*.csv' or all files '*'
files_to_upload = glob.glob(os.path.join(LOCAL_DESTINATION, '*'))

print(f"{len(files_to_upload)} files to upload.")

for local_file_path in files_to_upload:
    # Extract just the filename from the full path
    file_name = os.path.basename(local_file_path)

    # Construct the S3 object key (path in S3)
    s3_key = f"{S3_DES_FILE_PATH}/{file_name}" if S3_DES_FILE_PATH else file_name

    try:
        # Upload the file
        s3.upload_file(local_file_path, S3_DES_BUCKET_NAME, s3_key)
        print(f"Successfully uploaded {file_name} to s3://{S3_DES_BUCKET_NAME}/{s3_key}")
    except Exception as e:
        print(f"Error uploading {file_name}: {e}")

print("All uploads completed!")